# Studying the initial neutron star population

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib import rc
import matplotlib as mpl
from scipy.integrate import quad
import pypopsyn.simulator.constants as const
import pypopsyn.simulator.initial_position as ip

# Set `usetex=False' if you do not have LaTeX installed.
rc('text', usetex=False)
rc('font', family='serif')
mpl.rcParams['text.latex.preamble'] = [r"\usepackage{amsmath}"]

In [ ]:
rcParams["mathtext.fontset"] = "stix"
# rcParams["font.family"] = "Liberation serif"
rcParams["font.size"] = "22"
# rcParams['font.weight']='bold'
rcParams["figure.figsize"] = "8.0, 8.0"
rcParams["figure.autolayout"] = "False"

rcParams["axes.linewidth"] = "1.7"
rcParams["axes.labelpad"] = "15.0"
rcParams["axes.titlepad"] = "15.0"

rcParams["xtick.direction"] = "in"
rcParams["xtick.top"] = True
rcParams["xtick.major.pad"] = "10.0"
rcParams["xtick.minor.pad"] = "10.0"
rcParams["xtick.major.size"] = "10.0"
rcParams["xtick.major.width"] = "1.7"
rcParams["xtick.minor.size"] = "5.0"
rcParams["xtick.minor.width"] = "1.7"
rcParams["xtick.labelsize"] = "25"

rcParams["ytick.direction"] = "in"
rcParams["ytick.right"] = True
rcParams["ytick.major.pad"] = "10.0"
rcParams["ytick.minor.pad"] = "10.0"
rcParams["ytick.major.size"] = "10.0"
rcParams["ytick.major.width"] = "1.7"
rcParams["ytick.minor.size"] = "5.0"
rcParams["ytick.minor.width"] = "1.7"
rcParams["ytick.labelsize"] = "25"

Select an `initial_population.pkl.gz` file to import:

In [ ]:
data = pd.read_pickle("../examples/data/simulation_maxwell_sigma265_h018/initial_population.pkl.gz", compression="gzip")
data.head()

In [ ]:
age = data["age"]["[yr]"].to_numpy()
x = data["x"]["[kpc]"].to_numpy()
y = data["y"]["[kpc]"].to_numpy()
z = data["z"]["[kpc]"].to_numpy()
vk_r = data["vk_r"]["[kpc/yr]"].to_numpy() * const.KPC_TO_KM / const.YR_TO_S
vk_phi = data["vk_phi"]["[kpc/yr]"].to_numpy() * const.KPC_TO_KM / const.YR_TO_S
vk_z = data["vk_z"]["[kpc/yr]"].to_numpy() * const.KPC_TO_KM / const.YR_TO_S
v_orb = data["v_orb"]["[kpc/yr]"].to_numpy() * const.KPC_TO_KM / const.YR_TO_S

Top view of the galactic plane

In [ ]:
fig, ax = plt.subplots()

ax.plot(
    x,
    y,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=1,
    alpha=0.2,
    rasterized=True
)

ax.plot(0.0, 8.3, marker="o", color="tab:orange", markersize=6)
ax.set_xlabel(r"$x$ [kpc]")
ax.set_ylabel(r"$y$ [kpc]")
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-20.0, 20.0)

plt.show()

Side view of the galactic plane

In [ ]:
fig, ax = plt.subplots()

ax.plot(
    x,
    z,
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=1,
    alpha=0.3,
    rasterized=True,
)

ax.plot(0.0, 0.02, marker="o", color="tab:orange", markersize=6)
ax.set_xlabel(r"$x$ [kpc]")
ax.set_ylabel(r"$z$ [kpc]")
ax.set_xlim(-20.0, 20.0)
ax.set_ylim(-20.0, 20.0)

plt.show()

Histrogramming the pulsars radial position and comparing to underlying PDF

In [ ]:
def pdf_r(r: float) -> float:
    """
    The Milky Way's stellar radial density in the galactic plane according
    to eq. (15) of Yusifov & Küçük (2004).

    Args:
        r (float): distance from the galactic center in [kpc].

    Returns:
        float: stellar radial density in [1/kpc].
    """

    # Here we keep R_sun = 8.5 kpc for consistency with the results
    # of Yusifov & Küçük (2004)
    rsun = 8.5  # Sun's distance from the galactic center in [kpc].
    A = 37.6  # +- 1.90 [1/kpc^2]
    a = 1.64  # +-0.11
    b = 4.01  # +-0.24
    r1 = 0.55  # +- 0.10 [kpc]

    # Stellar surface density following eq. (15) of Yusifov & Küçük (2004).
    rho = (
        A
        * ((r + r1) / (rsun + r1)) ** a
        * np.exp(-b * (r - rsun) / (rsun + r1))
    )

    # Multiply the stellar surface density with the area element in polar coordinates.
    pdf_r = 2 * np.pi * r * rho

    return pdf_r

For normalization purposes, determine the area underneath the theoretical PDF curve:

In [ ]:
pdf_area = quad(pdf_r, 0, 100)[0]
print(pdf_area)

In [ ]:
r = np.sqrt(x**2 + y**2)
r_edges = np.linspace(0.0, 30.0, 51)

In [ ]:
fig, ax = plt.subplots()

ax.hist(
    r,
    bins=r_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label="Initial simulation",
    density=True
)
ax.plot(
    r_edges,
    pdf_r(r_edges)/pdf_area,
    linestyle="-",
    lw=4,
    color="tab:red",
    alpha=1,
    label="Theoretical YK04",
)
plt.xlabel(r"$r$ [kpc]")
plt.ylabel(r"Normalized radial PDF")
plt.xlim(0.0, 30.0)
plt.legend(frameon=False, loc=1)

plt.show()

Histrogramming the pulsars $z$ position and comparing to underlying PDF

In [ ]:
def pdf_z(z: float) -> float:
    """
    Probability density function for the height from the galactic equatorial plane
    according to eq. (2) in Gullon et al. (2014).

    Args:
        z (float): distance from the galactic plane in [kpc].

    Returns:
        float: distribution of stars per kpc in z direction.
    """

    # We use an exponential distribution as given by Wainscoat et al. (1992)
    # and choose a mean scale height characteristic for a young distribution as
    # obtained by Gullon et al. (2014).

    h_c = 0.18
    pdf_z = 1.0 / h_c * np.exp(-z / h_c)

    return pdf_z

In [ ]:
z_edges = np.linspace(0.0, 5.0, 51)

fig, ax = plt.subplots()

ax.hist(
    z,
    bins=z_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label="Initial simulation",
    density=True
)
ax.plot(
    z_edges,
    pdf_z(z_edges),
    linestyle="-",
    lw=4,
    color="tab:red",
    alpha=1,
    label="theoretical exp",
)
plt.xlabel(r"$z$ [kpc]")
plt.ylabel(r"Normalized height PDF")
plt.xlim(0.0, 5.0)
plt.legend(frameon=False, loc=1)

plt.show()

Histogrammed kick velocity components

In [ ]:
vk_edges = np.linspace(-1500.,1500.,51) 

fig, ax = plt.subplots() 

ax.hist(
    vk_r,
    bins=x_bins,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"$v_{{\rm k,}r}$",
)
ax.hist(
    vk_phi,
    bins=x_bins,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"$v_{{\rm k,}\phi}$",
)
ax.hist(
    vk_z,
    bins=x_bins,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    alpha=1,
    label=r"$v_{{\rm k,}z}$",
)
ax.set_xlabel(r"Kick velocity components [km s$^{-1}$]")
ax.set_ylabel(r"Number of NSs")
plt.legend(frameon=False, loc=0)

plt.show()

Distribution of the total kick velocity magnitude

In [ ]:
def pdf_kick_velocity_maxwell(v: float) -> float:
    """
    Maxwell probability density function for the neutron stars' initial kick
    velocity magnitude following Hobbs et al. (2005).

    Args:
        v (float): initial kick velocity magnitude in [km/s].

    Returns:
        float: stellar kick velocity distribution in [1/(km/s)].
    """
    sigma = 265.
    pdf_vk = (
        np.sqrt(2 / np.pi)
        * v ** 2
        / (sigma ** 3)
        * np.exp(-(v ** 2) / (2 * sigma ** 2))
    )

    return pdf_vk

In [ ]:
vk_tot = np.sqrt(vk_r**2 + vk_phi**2 + vk_z**2)

vk_edges = np.linspace(0,1500.,51)  

fig, ax = plt.subplots()

ax.hist(
    vk_tot,
    bins=vk_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated",
    density=True
)
ax.plot(
    vk_edges,
    pdf_kick_velocity_maxwell(vk_edges),
    linestyle="-",
    lw=4,
    color="tab:red",
    alpha=1,
    label=r"Maxwell $\sigma = 265$ km s$^{-1}$",
)
ax.set_xlabel(r"Kick velocity magnitude [km s$^{-1}$]")
ax.set_ylabel(r"Number of NSs")
plt.legend(frameon=False, loc=0)

plt.show()

Orbital (angular) velocity due to galactic potential

In [ ]:
r = np.sqrt(x**2 + y**2)

fig, ax = plt.subplots()

ax.plot(
    r,
    abs(v_orb),
    linestyle="None",
    marker="o",
    color="tab:blue",
    markersize=1,
    alpha=0.3,
    rasterized=True,
)

ax.set_xlabel(r"$r$ [kpc]")
ax.set_ylabel(r"$v_{\rm orb}$ [km s$^{-1}$]")

plt.show()

Histogrammed velocity components

In [ ]:
v_edges = np.linspace(-1500.,1500.,51)  

fig, ax = plt.subplots()

ax.hist(
    vk_r,
    bins=v_edges ,
    histtype="step",
    edgecolor="tab:red",
    lw=4,
    alpha=1,
    label=r"$v_r$",
)
ax.hist(
    vk_phi + v_orb,
    bins=v_edges ,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"$v_{\phi}$",
)
ax.hist(
    vk_z,
    bins=v_edges ,
    histtype="step",
    edgecolor="tab:green",
    lw=4,
    alpha=1,
    label=r"$v_z$",
)
ax.set_xlabel(r"Velocity components [km s$^{-1}$]")
ax.set_ylabel(r"Number of NSs")
plt.legend(frameon=False, loc=0)

plt.show()

Distribution of the total velocity magnitude

In [ ]:
v_tot = np.sqrt(vk_r**2 + (vk_phi+v_orb)**2 + vk_z**2)

v_edges = np.linspace(0.,1500.,51) 

fig, ax = plt.subplots()  

ax.hist(
    v_tot,
    bins=v_edges,
    histtype="step",
    edgecolor="tab:blue",
    lw=4,
    alpha=1,
    label=r"Simulated",
    density=True
)
ax.plot(
    v_edges ,
    pdf_kick_velocity_maxwell(v_edges ),
    linestyle="-",
    lw=4,
    color="tab:red",
    alpha=1,
    label=r"Maxwell $\sigma = 265$ km s$^{-1}$",
)
ax.set_xlabel(r"3D velocity magnitude [km s$^{-1}$]")
ax.set_ylabel(r"Number of NSs")
plt.legend(frameon=False, loc=0)

plt.show()